In [ ]:
# Task: Date Alignment

import pandas as pd

# Load news dataset
news_df = pd.read_csv('../data/raw/raw_analyst_ratings.csv') 

# Convert to datetime objects using 'mixed' to handle the different strings in the file
news_df['date'] = pd.to_datetime(news_df['date'], format='mixed', dayfirst=False, errors='coerce', utc=True)

# Convert to naive datetime (removes the +00:00) so it's easier to merge with stock data
news_df['date'] = news_df['date'].dt.tz_localize(None)

# Check the data type
print(f"Column Data Type: {news_df['date'].dtype}")

# Look at the head to see the standardized YYYY-MM-DD format
print(news_df['date'].head(12))

# Check if any dates became 'NaT' (Not a Time)
failed_conversions = news_df['date'].isna().sum()

if failed_conversions == 0:
    print(f"Success! All {len(news_df)} rows were converted to a uniform format.")
else:
    print(f"Warning: {failed_conversions} rows could not be parsed.")


Column Data Type: datetime64[ns]
0    2020-06-05 14:30:54
1    2020-06-03 14:45:20
2    2020-05-26 08:30:07
3    2020-05-22 16:45:06
4    2020-05-22 15:38:59
5    2020-05-22 15:23:25
6    2020-05-22 13:36:20
7    2020-05-22 13:07:04
8    2020-05-22 12:37:59
9    2020-05-22 12:06:17
10   2020-05-22 00:00:00
11   2020-05-22 00:00:00
Name: date, dtype: datetime64[ns]
Success! All 1407328 rows were converted to a uniform format.


In [28]:
# Weekend Alignment Function
def align_to_trading_day(df):
    """
    Shifts Saturday news (+2 days) and Sunday news (+1 day) to Monday.
    This ensures news sentiment can be correlated with market opens.
    """
    # .weekday(): Monday=0, Tuesday=1, ... Saturday=5, Sunday=6
    df['trading_date'] = df['date'].apply(
        lambda x: x + pd.Timedelta(days=2) if x.weekday() == 5 
        else (x + pd.Timedelta(days=1) if x.weekday() == 6 else x)
    )
    # Strip time so we can merge on 'Date' only (YYYY-MM-DD)
    df['trading_date'] = df['trading_date'].dt.date
    return df

news_df = align_to_trading_day(news_df)

# See the results
print("Alignment Verification:")
print(news_df[['date', 'trading_date']].head(12))

# Check for weekends
print("\n--- Count of headlines per day of the week with the weekends added on Monday (0=Mon, 6=Sun) ---")
# This should show 0, 1, 2, 3, 4. If 5 and 6 are 0, it worked!
print(pd.to_datetime(news_df['trading_date']).dt.weekday.value_counts())


Alignment Verification:
                  date trading_date
0  2020-06-05 14:30:54   2020-06-05
1  2020-06-03 14:45:20   2020-06-03
2  2020-05-26 08:30:07   2020-05-26
3  2020-05-22 16:45:06   2020-05-22
4  2020-05-22 15:38:59   2020-05-22
5  2020-05-22 15:23:25   2020-05-22
6  2020-05-22 13:36:20   2020-05-22
7  2020-05-22 13:07:04   2020-05-22
8  2020-05-22 12:37:59   2020-05-22
9  2020-05-22 12:06:17   2020-05-22
10 2020-05-22 00:00:00   2020-05-22
11 2020-05-22 00:00:00   2020-05-22

--- Count of headlines per day of the week with the weekends added on Monday (0=Mon, 6=Sun) ---
trading_date
3    302619
2    300922
1    296505
0    289364
4    217918
Name: count, dtype: int64


In [31]:
# Handling Holidays
from pandas.tseries.holiday import USFederalHolidayCalendar

def align_to_trading_day_with_holidays(df):
    # 1. Identify US Holidays
    cal = USFederalHolidayCalendar()
    holidays = cal.holidays(start=df['date'].min(), end=df['date'].max())
    
    df['trading_date'] = pd.to_datetime(df['date']).dt.date
    
    # 2. Loop to push dates forward if they hit a weekend or holiday
    def get_next_trading_day(d):
        d = pd.to_datetime(d)
        # While the day is a weekend OR a holiday, add 1 day
        while d.weekday() >= 5 or d in holidays:
            d += pd.Timedelta(days=1)
        return d.date()

    # Note: This takes a moment on 1.4M rows
    print("Aligning dates to next trading day (including holidays)...")
    df['trading_date'] = df['trading_date'].apply(get_next_trading_day)
    return df

news_df = align_to_trading_day_with_holidays(news_df)

# Create a list of some major holidays within your data range
test_holidays = ['2020-01-01', '2019-07-04', '2020-12-25']

# Filter the dataframe to see headlines from those specific days
holiday_check = news_df[news_df['date'].dt.strftime('%Y-%m-%d').isin(test_holidays)]

if not holiday_check.empty:
    print("Holiday Alignment Verification:")
    # Show the original date vs the trading date
    print(holiday_check[['date', 'trading_date']].head(10))
else:
    print("No headlines found on those specific dates, trying a broader search...")


# Alternative check if any trading_date matches a holiday (it shouldn't!)
from pandas.tseries.holiday import USFederalHolidayCalendar
cal = USFederalHolidayCalendar()
holidays = cal.holidays(start=news_df['date'].min(), end=news_df['date'].max())
    
    # This should be 0 if the code worked
invalid_dates = news_df[pd.to_datetime(news_df['trading_date']).isin(holidays)]
print(f"Number of headlines still sitting on a holiday: {len(invalid_dates)}")


Aligning dates to next trading day (including holidays)...
Holiday Alignment Verification:
                      date trading_date
42310  2020-01-01 00:00:00   2020-01-02
160573 2020-01-01 00:00:00   2020-01-02
174389 2020-01-01 00:00:00   2020-01-02
185183 2020-01-01 00:00:00   2020-01-02
185518 2020-01-01 15:47:10   2020-01-02
289511 2020-01-01 00:00:00   2020-01-02
380833 2020-01-01 00:00:00   2020-01-02
428226 2020-01-01 00:00:00   2020-01-02
546613 2020-01-01 00:00:00   2020-01-02
555788 2020-01-01 00:00:00   2020-01-02
Number of headlines still sitting on a holiday: 0


In [33]:
# Task: Sentiment Analysis

# Initialize VADER and Progress Tracking

import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from tqdm import tqdm

# Download the lexicon (the dictionary of 'feeling' words)
nltk.download('vader_lexicon')

# Initialize the analyzer
sia = SentimentIntensityAnalyzer()

# This line allows pandas to use .progress_apply()
tqdm.pandas()

# Run the Sentiment Scoring using the code below
print("Analyzing sentiment for 1.4M headlines...")

# We use 'str(x)' to ensure any non-text headlines don't crash the script
news_df['sentiment_score'] = news_df['headline'].progress_apply(
    lambda x: sia.polarity_scores(str(x))['compound']
)


[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\nemsa\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


Analyzing sentiment for 1.4M headlines...


100%|██████████| 1407328/1407328 [03:36<00:00, 6497.06it/s]


In [35]:
# Categorize the Scores

def categorize_sentiment(score):
    if score >= 0.05:
        return 'positive'
    elif score <= -0.05:
        return 'negative'
    else:
        return 'neutral'

news_df['sentiment_class'] = news_df['sentiment_score'].apply(categorize_sentiment)

# Check the distribution
print("\nSentiment Distribution:")
print(news_df['sentiment_class'].value_counts())


Sentiment Distribution:
sentiment_class
neutral     741194
positive    441858
negative    224276
Name: count, dtype: int64


In [36]:
# Create the directory if it doesn't exist
import os
os.makedirs('../data/processed', exist_ok=True)

# Save to CSV
output_path = '../data/processed/aligned_news_sentiment.csv'
news_df.to_csv(output_path, index=False)

print(f"Success! Final dataset saved to {output_path}")

Success! Final dataset saved to ../data/processed/aligned_news_sentiment.csv
